In [8]:
# ==============================================================================
# 03. Risk Bucketing & Expected Loss Calculations
# ==============================================================================
import pandas as pd
import numpy as np
import os

In [9]:
os.makedirs('../outputs/reports', exist_ok=True)

In [10]:
# 1. Load Test Predictions
results_df = pd.read_csv('../data/processed/test_predictions.csv')

In [11]:
# 2. Risk Bucketing (Deciles)
# Tier 1 = Lowest Risk, Tier 10 = Highest Risk
results_df['risk_tier'] = pd.qcut(results_df['predicted_pd'], q=10, labels=range(1, 11))

In [12]:
# 3. Monotonicity Validation & EL Calculation
# Assumptions derived from business context: LGD = 45%, Avg EAD = $15,000
LGD = 0.45      
AVG_EAD = 15000 

risk_table = results_df.groupby('risk_tier', observed=False).agg(
    loan_count=('actual_default', 'count'),
    observed_default_rate=('actual_default', 'mean'),
    avg_predicted_pd=('predicted_pd', 'mean')
).reset_index()

risk_table['expected_loss_per_loan'] = risk_table['avg_predicted_pd'] * LGD * AVG_EAD
risk_table['tier_total_expected_loss'] = risk_table['expected_loss_per_loan'] * risk_table['loan_count']

In [13]:
# 4. Format and Display
formatted_table = risk_table.copy()
formatted_table['observed_default_rate'] = formatted_table['observed_default_rate'].map("{:.2%}".format)
formatted_table['avg_predicted_pd'] = formatted_table['avg_predicted_pd'].map("{:.2%}".format)
formatted_table['expected_loss_per_loan'] = formatted_table['expected_loss_per_loan'].map("${:,.2f}".format)
formatted_table['tier_total_expected_loss'] = formatted_table['tier_total_expected_loss'].map("${:,.0f}".format)

print("--- Basel-Aligned Risk Grading & Expected Loss ---")
display(formatted_table)

--- Basel-Aligned Risk Grading & Expected Loss ---


,risk_tier,loan_count,observed_default_rate,avg_predicted_pd,expected_loss_per_loan,tier_total_expected_loss
0,1,422,1.42%,3.34%,$225.42,"$95,128"
1,2,422,4.27%,4.22%,$284.89,"$120,222"
2,3,422,4.74%,5.04%,$340.43,"$143,661"
3,4,421,6.41%,6.30%,$424.94,"$178,901"
4,5,422,6.40%,7.92%,$534.27,"$225,463"
5,6,422,8.77%,9.60%,$648.14,"$273,515"
6,7,421,11.88%,11.44%,$772.15,"$325,074"
7,8,422,13.51%,13.91%,$938.91,"$396,218"
8,9,422,22.51%,18.68%,"$1,261.02","$532,150"
9,10,422,34.12%,32.56%,"$2,197.68","$927,419"


In [14]:
# 5. Export Report
risk_table.to_csv('../outputs/reports/risk_tier_expected_loss.csv', index=False)
print("Expected Loss report saved to '../outputs/reports/risk_tier_expected_loss.csv'")

Expected Loss report saved to '../outputs/reports/risk_tier_expected_loss.csv'
